In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [3]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,3,15619304,Onio,502,France,Female,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,4,15701354,Boni,699,France,Female,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43.0,2,125510.82,1,NaN,1.0,79084.10,0


In [4]:
print(data.isnull().sum())
print(data.shape)

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          1
Gender             0
Age                1
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          1
IsActiveMember     1
EstimatedSalary    0
Exited             0
dtype: int64
(10002, 14)


In [5]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data
# 1. duplicates teeseyandi
data = data.drop_duplicates()
data = data.dropna()
data = data.reset_index(drop=True)

print(data.isnull().sum())
print(data.shape)


CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64
(9996, 11)


In [6]:
print(data.isnull().sum().sum()) 

0


In [7]:
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,608,Spain,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,502,France,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,699,France,0,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,645,Spain,1,44.0,8,113755.78,2,1.0,0.0,149756.71,1
...,...,...,...,...,...,...,...,...,...,...,...
9991,771,France,1,39.0,5,0.00,2,1.0,0.0,96270.64,0
9992,516,France,1,35.0,10,57369.61,1,1.0,1.0,101699.77,0
9993,709,France,0,36.0,7,0.00,1,0.0,1.0,42085.58,1
9994,772,Germany,1,42.0,3,75075.31,2,1.0,0.0,92888.52,1


In [8]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9996 stored elements and shape (9996, 3)>

In [9]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [10]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [11]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9991,1.0,0.0,0.0
9992,1.0,0.0,0.0
9993,1.0,0.0,0.0
9994,0.0,1.0,0.0


In [12]:
#combine all the columns with original data
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42.0,2,0.00,1,1.0,1.0,101348.88,1,1.0,0.0,0.0
1,608,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0,0.0,0.0,1.0
2,502,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1,1.0,0.0,0.0
3,699,0,39.0,1,0.00,2,0.0,0.0,93826.63,0,1.0,0.0,0.0
4,645,1,44.0,8,113755.78,2,1.0,0.0,149756.71,1,0.0,0.0,1.0


In [13]:
#save the encoders and scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)








In [14]:
##divide the dataset into independent and dependent features
X=data.drop('Exited',axis=1)
y=data['Exited']
#split the data into training and testing tests
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
#Scale the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [15]:
X_train

array([[-1.24763692, -1.07916228,  0.86412027, ...,  0.99650436,
        -0.578313  , -0.57369185],
       [ 0.00788859, -1.07916228, -0.37178168, ..., -1.0035079 ,
         1.72916742, -0.57369185],
       [ 0.77573064,  0.92664469,  0.00849584, ...,  0.99650436,
        -0.578313  , -0.57369185],
       ...,
       [-0.13737882, -1.07916228,  0.48384275, ..., -1.0035079 ,
        -0.578313  ,  1.74309605],
       [-0.58355731,  0.92664469, -0.08657354, ...,  0.99650436,
        -0.578313  , -0.57369185],
       [ 0.01826484, -1.07916228, -0.46685106, ...,  0.99650436,
        -0.578313  , -0.57369185]])

In [16]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


In [18]:
##Build ANN Model
model=Sequential([
  Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ##HL1 connected with input layer
  Dense(32,activation='relu'),##HL2
  Dense(1,activation='sigmoid')##output layer
])

In [19]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [20]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.001)
tensorflow.keras.losses.BinaryCrossentropy()
#compile the model 
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [21]:
##Set up the TensorBoard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir="logs/fit/" +datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)


In [22]:
##set up early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [23]:
##Train the Model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 5s 7ms/step - loss: 0.4455 - accuracy: 0.8065 - val_loss: 0.4011 - val_accuracy: 0.8265
Epoch 2/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3725 - accuracy: 0.8489 - val_loss: 0.3658 - val_accuracy: 0.8450
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3499 - accuracy: 0.8579 - val_loss: 0.3585 - val_accuracy: 0.8430
Epoch 4/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3404 - accuracy: 0.8602 - val_loss: 0.3564 - val_accuracy: 0.8540
Epoch 5/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3350 - accuracy: 0.8631 - val_loss: 0.3525 - val_accuracy: 0.8505
Epoch 6/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3317 - accuracy: 0.8644 - val_loss: 0.3517 - val_accuracy: 0.8545
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3292 - accuracy: 0.8634 - val_loss: 0.3530 - val_accuracy: 0.85

In [24]:
model.save('model.h5')

d:\GEN AI\room\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [25]:
##LOad Tensorboard extension
%load_ext tensorboard

In [26]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 23580), started 2 days, 9:15:43 ago. (Use '!kill 23580' to kill it.)

In [ ]:
#Load pickle file
